# Estimacion de Plazo de Entrega - Olist

Analisis completo del caso 3 del proyecto: estimacion del plazo de entrega usando modelos de regresion,
identificacion de pedidos con riesgo de retraso y visualizacion de resultados.

## 1. Configuracion del entorno

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.base import clone

from src.data.load_data import load_csv
from src.data.split import split_dataset
from src.delivery.dataset import build_delivery_dataset
from src.delivery.evaluation import (
    evaluate_delivery_models,
    get_delivery_feature_importance,
    identify_orders_with_delay_risk,
)
from src.delivery.features import create_delivery_features
from src.delivery.training import train_and_validate_delivery_models
from src.pipelines.delivery_pipeline import run_delivery_pipeline

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 60)

## 2. Analisis Exploratorio de Datos (EDA)

Cargamos los datasets base de ordenes, items, clientes y vendedores para entender el contexto del problema.
En este caso, el feature de carrier se aproxima con `shipping_limit_date`, ya que Olist no incluye un carrier_id explicito.

In [ ]:
orders = load_csv("olist_orders_dataset.csv")
order_items = load_csv("olist_order_items_dataset.csv")
customers = load_csv("olist_customers_dataset.csv")
sellers = load_csv("olist_sellers_dataset.csv")

print("Dimensiones de los datasets:")
for name, df in [
    ("Orders", orders),
    ("Order Items", order_items),
    ("Customers", customers),
    ("Sellers", sellers),
]:
    print(f"  {name}: {df.shape[0]} filas x {df.shape[1]} columnas")

print(f"\nOrdenes entregadas: {(orders['order_status'] == 'delivered').sum()}")

In [ ]:
delivery_data = build_delivery_dataset()
print(f"Dataset de delivery: {delivery_data.shape[0]} pedidos x {delivery_data.shape[1]} columnas")
print(f"Tasa historica de retraso: {delivery_data['is_late'].mean():.2%}")
delivery_data.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

delivery_data['delivery_days'].plot(kind='hist', bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('Distribucion del plazo de entrega')
axes[0].set_xlabel('Dias')

delivery_data['is_late'].value_counts().plot(kind='bar', ax=axes[1], color=['teal', 'coral'])
axes[1].set_title('Pedidos entregados en plazo vs retrasados')
axes[1].set_xticklabels(['En plazo (0)', 'Tarde (1)'], rotation=0)
axes[1].set_ylabel('Cantidad de pedidos')

plt.tight_layout()
plt.show()

## 3. Feature Engineering

Se generan features con ubicacion del cliente y vendedor, categoria principal del pedido,
peso, volumen, precio, freight, distancia geografica y proxy de carrier.

In [ ]:
features, target, feature_names = create_delivery_features(delivery_data)
print(f"Features: {features.shape[0]} filas x {features.shape[1]} columnas")
print(f"Target promedio: {target.mean():.2f} dias")
print(f"Primeras 20 features: {feature_names[:20]}")

## 4. Entrenamiento de Modelos

Se entrenan los modelos pedidos para el bloque 3:
- Linear Regression
- Random Forest Regressor
- Gradient Boosting Regressor
- SVR
- k-NN Regressor

In [ ]:
X_train, X_test, y_train, y_test = split_dataset(features, target, use_stratify=False)
print(f"Train: {X_train.shape[0]} muestras")
print(f"Test: {X_test.shape[0]} muestras")

trained_models, cv_results = train_and_validate_delivery_models(X_train, y_train)
cv_results

## 5. Evaluacion en Test Set

Las metricas principales para este caso son:
- MAE
- RMSE
- R2

In [ ]:
metrics_df, predictions = evaluate_delivery_models(trained_models, X_test, y_test)
metrics_df

In [ ]:
best_model_name = metrics_df.index[0]
best_model = trained_models[best_model_name]
best_predictions = predictions[best_model_name].copy()
best_predictions.insert(0, 'order_id', delivery_data.loc[X_test.index, 'order_id'].values)

print(f"Mejor modelo: {best_model_name}")
best_predictions.head()

## 6. Riesgo de Retraso

Se identifica riesgo de retraso comparando el plazo estimado por el mejor modelo
contra el plazo prometido al cliente.

In [ ]:
production_model = clone(best_model)
production_model.fit(features, target)

risk_orders = identify_orders_with_delay_risk(
    production_model,
    features,
    delivery_data[['order_id', 'delivery_days', 'promised_delivery_days', 'delay_days', 'is_late']],
)

print(f"Pedidos con riesgo estimado de retraso: {risk_orders['delay_risk'].sum()}")
risk_orders.head(10)

## 7. Visualizaciones

Visualizamos la relacion entre valores reales y predichos, los residuales y la importancia de variables.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(best_predictions['actual_delivery_days'], best_predictions['predicted_delivery_days'], alpha=0.45, s=20, color='steelblue')
min_value = min(best_predictions['actual_delivery_days'].min(), best_predictions['predicted_delivery_days'].min())
max_value = max(best_predictions['actual_delivery_days'].max(), best_predictions['predicted_delivery_days'].max())
ax.plot([min_value, max_value], [min_value, max_value], 'r--', linewidth=2)
ax.set_title(f'Predicho vs real - {best_model_name}')
ax.set_xlabel('Delivery real (dias)')
ax.set_ylabel('Delivery predicho (dias)')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(best_predictions['predicted_delivery_days'], best_predictions['residual'], alpha=0.45, s=20, color='coral')
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Residuales vs prediccion')
axes[0].set_xlabel('Delivery predicho (dias)')
axes[0].set_ylabel('Residual')

sns.histplot(best_predictions['residual'], bins=30, kde=True, ax=axes[1], color='teal')
axes[1].set_title('Distribucion de residuales')
axes[1].set_xlabel('Residual')

plt.tight_layout()
plt.show()

In [ ]:
importance_data = get_delivery_feature_importance(trained_models, feature_names)

for model_name, importance in importance_data.items():
    print(f"\nTop 10 features - {model_name}")
    display(importance.head(10).to_frame('importance'))

## 8. Ejecucion End-to-End del Pipeline

Esta celda ejecuta el flujo productivo completo y deja guardados los artefactos del bloque 3.

In [ ]:
trained_models, metrics_df, risk_orders = run_delivery_pipeline()

## 9. Conclusiones

- Se construyo un pipeline de regresion para estimar plazo de entrega a nivel de pedido.
- Se incorporaron features de ubicacion, categoria, volumen, freight, peso y distancia geografica.
- Se evaluaron multiples modelos con MAE, RMSE y R2.
- Se identificaron pedidos con riesgo de retraso comparando el plazo estimado contra el prometido.
- Se generaron visualizaciones para interpretar ajuste, residuales e importancia de variables.